# Trenowanie modelu rozpoznawania ruchu



In [ ]:
from pathlib import Path
import sys
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'data').exists():
    # Działa także, gdy notebook zostanie uruchomiony z głównego folderu projektu.
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from imu_pipeline import fuse_raw_sensor_signals, fuse_sensor_features, read_ximu_csv, sampling_rate_hz
from build_training_dataset import FEATURE_SIGNALS, label_for_window, window_features

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifest.csv'
ANNOTATIONS_PATH = PROJECT_ROOT / 'data' / 'annotations.csv'
DATASET_PATH = PROJECT_ROOT / 'data' / 'training_windows.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_classical_stimming.joblib'
REPORTS_DIR = PROJECT_ROOT / 'reports'
print('Folder projektu:', PROJECT_ROOT)

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
annotations = pd.read_csv(ANNOTATIONS_PATH)

required_manifest = {'participant_id', 'session_id', 'wrist_path', 'lumbar_path', 'fully_annotated'}
required_annotations = {'session_id', 'start_s', 'end_s', 'label'}
assert not (required_manifest - set(manifest.columns)), 'Brakuje kolumn w manifest.csv'
assert not (required_annotations - set(annotations.columns)), 'Brakuje kolumn w annotations.csv'
assert manifest['participant_id'].nunique() >= 3, 'Potrzebne są dane co najmniej 3 osób.'

print('Liczba sesji:', len(manifest))
print('Liczba osób:', manifest['participant_id'].nunique())
display(manifest[['participant_id', 'session_id', 'lumbar_time_offset_s']])
display(annotations['label'].value_counts().rename_axis('etykieta').to_frame('liczba_przedziałów'))

In [ ]:
WINDOW_SECONDS = 3.0
HOP_SECONDS = 1.0
rows = []

for _, session in manifest.iterrows():
    wrist_path = PROJECT_ROOT / session['wrist_path']
    lumbar_path = PROJECT_ROOT / session['lumbar_path']
    if not wrist_path.exists() or not lumbar_path.exists():
        raise FileNotFoundError(f"Brak pliku w sesji {session['session_id']}")

    wrist = read_ximu_csv(wrist_path.read_bytes())
    lumbar = read_ximu_csv(lumbar_path.read_bytes())
    offset = float(session.get('lumbar_time_offset_s', 0.0))
    fused = fuse_sensor_features(wrist, lumbar, lumbar_time_offset_s=offset)
    session_labels = annotations[annotations['session_id'] == session['session_id']]
    rows.extend(window_features(fused, session, session_labels, WINDOW_SECONDS, HOP_SECONDS))

dataset = pd.DataFrame(rows)
dataset = dataset[dataset['label'] != 'unknown'].reset_index(drop=True)
DATASET_PATH.parent.mkdir(exist_ok=True)
dataset.to_csv(DATASET_PATH, index=False)
print(f'Zapisano {len(dataset)} okien: {DATASET_PATH}')
display(dataset['label'].value_counts().rename_axis('klasa').to_frame('liczba_okien'))

In [ ]:
METADATA_COLUMNS = {'participant_id', 'session_id', 'window_start_s', 'window_end_s', 'label'}
feature_columns = [c for c in dataset.columns if c not in METADATA_COLUMNS]
X = dataset[feature_columns]
y = dataset['label']
groups = dataset['participant_id']

print('Liczba cech:', len(feature_columns))
print('Liczba okien:', len(X))
print('Liczba osób w walidacji:', groups.nunique())
display(pd.DataFrame({'cecha': feature_columns}))

In [ ]:
# Cztery modele porównywane na identycznych oknach i identycznym podziale po osobach.
# Skalowanie jest konieczne dla Logistic Regression i SVM.
models_to_compare = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)),
    ]),
    'SVM RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(C=3.0, kernel='rbf', gamma='scale', class_weight='balanced')),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=400, class_weight='balanced_subsample', min_samples_leaf=2,
        n_jobs=-1, random_state=42,
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=400, class_weight='balanced', min_samples_leaf=2,
        n_jobs=-1, random_state=42,
    ),
}

splitter = LeaveOneGroupOut()
all_predictions = {}
all_reports = {}
comparison_rows = []

for model_name, base_model in models_to_compare.items():
    print(f'\n=== {model_name} ===')
    start_time = perf_counter()
    predictions = pd.Series(index=dataset.index, dtype='object')
    for round_no, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
        tested_person = groups.iloc[test_idx].iloc[0]
        print(f'Runda {round_no}/{groups.nunique()}: test dla {tested_person}')
        # Tworzony jest nowy model w każdej rundzie, aby testowana osoba nie trafiła do treningu.
        model = clone(base_model)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions.iloc[test_idx] = model.predict(X.iloc[test_idx])

    elapsed_s = perf_counter() - start_time
    report = pd.DataFrame(classification_report(y, predictions, output_dict=True, zero_division=0)).transpose()
    all_predictions[model_name] = predictions
    all_reports[model_name] = report
    comparison_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(y, predictions),
        'macro_f1': report.loc['macro avg', 'f1-score'],
        'weighted_f1': report.loc['weighted avg', 'f1-score'],
        'czas_treningu_s': elapsed_s,
    })

comparison = pd.DataFrame(comparison_rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(comparison.style.format({'accuracy': '{:.3f}', 'macro_f1': '{:.3f}', 'weighted_f1': '{:.3f}', 'czas_treningu_s': '{:.1f}'}))
best_model_name = comparison.loc[0, 'model']
print(f'Najlepszy model według macro F1: {best_model_name}')

In [ ]:
classes = sorted(y.unique())
best_predictions = all_predictions[best_model_name]
best_report = all_reports[best_model_name]
matrix = pd.DataFrame(confusion_matrix(y, best_predictions, labels=classes), index=classes, columns=classes)
matrix.index.name = 'prawdziwa_klasa'
matrix.columns.name = 'przewidziana_klasa'

REPORTS_DIR.mkdir(exist_ok=True)
comparison.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
best_report.to_csv(REPORTS_DIR / 'classification_report.csv')
matrix.to_csv(REPORTS_DIR / 'confusion_matrix.csv')
pd.DataFrame({
    'session_id': dataset['session_id'],
    'participant_id': groups,
    'window_start_s': dataset['window_start_s'],
    'window_end_s': dataset['window_end_s'],
    'actual_label': y,
    'predicted_label': best_predictions,
}).to_csv(REPORTS_DIR / 'out_of_sample_predictions.csv', index=False)

display(best_report[['precision', 'recall', 'f1-score', 'support']])
display(matrix)
print('Zapisano raporty w:', REPORTS_DIR)

In [ ]:
final_model = clone(models_to_compare[best_model_name])
final_model.fit(X, y)
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({
    'model': final_model,
    'model_name': best_model_name,
    'feature_columns': feature_columns,
    'classes': classes,
}, MODEL_PATH)
print(f'Zapisano najlepszy model ({best_model_name}):', MODEL_PATH)

## Modele głębokie: 1D CNN i TCN

W odróżnieniu od modeli klasycznych oba modele dostają surowy przebieg 12 kanałów IMU. Walidacja nadal jest wykonywana po osobach. Domyślne 8 epok ustawiono celowo mało, aby ograniczyć przeuczenie przy 15 osobach.

In [ ]:
import copy
import os

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SAMPLES = 300  # 3 s przy około 100 Hz
DEEP_EPOCHS = 4  # mały zbiór i trening CPU: ograniczamy ryzyko przeuczenia oraz czas obliczeń
BATCH_SIZE = 256
RAW_CHANNELS = [
    'wrist_acc_x', 'wrist_acc_y', 'wrist_acc_z', 'wrist_gyro_x', 'wrist_gyro_y', 'wrist_gyro_z',
    'lumbar_acc_x', 'lumbar_acc_y', 'lumbar_acc_z', 'lumbar_gyro_x', 'lumbar_gyro_y', 'lumbar_gyro_z',
]
print('Urządzenie:', DEVICE)

In [ ]:
def resample_window(values, target_samples=TARGET_SAMPLES):
    """Zmienia liczbę próbek w oknie, zachowując wszystkie 12 kanałów."""
    old_axis = np.linspace(0, 1, len(values))
    new_axis = np.linspace(0, 1, target_samples)
    return np.stack([np.interp(new_axis, old_axis, values[:, channel]) for channel in range(values.shape[1])], axis=1)

sequence_rows, sequence_labels, sequence_groups, sequence_sessions = [], [], [], []
for _, session in manifest.iterrows():
    wrist_path = PROJECT_ROOT / session['wrist_path']
    lumbar_path = PROJECT_ROOT / session['lumbar_path']
    wrist = read_ximu_csv(wrist_path.read_bytes())
    lumbar = read_ximu_csv(lumbar_path.read_bytes())
    raw = fuse_raw_sensor_signals(wrist, lumbar, float(session.get('lumbar_time_offset_s', 0.0)))
    rate = sampling_rate_hz(raw)
    window_samples, hop_samples = round(WINDOW_SECONDS * rate), round(HOP_SECONDS * rate)
    session_labels = annotations[annotations['session_id'] == session['session_id']]
    for start_index in range(0, len(raw) - window_samples + 1, hop_samples):
        chunk = raw.iloc[start_index:start_index + window_samples]
        start_s, end_s = float(chunk['time_s'].iloc[0]), float(chunk['time_s'].iloc[-1])
        label = label_for_window(session_labels, start_s, end_s, bool(session['fully_annotated']))
        if label == 'unknown':
            continue
        sequence_rows.append(resample_window(chunk[RAW_CHANNELS].to_numpy(dtype=np.float32)))
        sequence_labels.append(label)
        sequence_groups.append(session['participant_id'])
        sequence_sessions.append(session['session_id'])

X_sequence = np.stack(sequence_rows).astype(np.float32)
deep_classes = sorted(set(sequence_labels))
class_to_index = {label: index for index, label in enumerate(deep_classes)}
y_sequence = np.array([class_to_index[label] for label in sequence_labels], dtype=np.int64)
deep_groups = np.array(sequence_groups)
print('Kształt danych sekwencyjnych:', X_sequence.shape, '(okna, próbki, kanały)')
print('Klasy:', deep_classes)

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, channels, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 16, kernel_size=7, padding=3), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.15),
            nn.Conv1d(16, 24, kernel_size=5, padding=2), nn.BatchNorm1d(24), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.20),
            nn.Conv1d(24, 24, kernel_size=3, padding=1), nn.BatchNorm1d(24), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(24, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        padding = 2 * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation)
        self.norm1, self.norm2 = nn.BatchNorm1d(out_channels), nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(0.20)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.trim = padding
    def forward(self, x):
        out = self.dropout(torch.relu(self.norm1(self.conv1(x))))
        out = out[:, :, :-self.trim] if self.trim else out
        out = self.dropout(torch.relu(self.norm2(self.conv2(out))))
        out = out[:, :, :-self.trim] if self.trim else out
        return torch.relu(out + self.downsample(x))

class TCN(nn.Module):
    def __init__(self, channels, n_classes):
        super().__init__()
        self.network = nn.Sequential(
            TemporalBlock(channels, 16, dilation=1),
            TemporalBlock(16, 24, dilation=2),
        )
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(24, n_classes))
    def forward(self, x):
        return self.classifier(self.network(x))

In [ ]:
def train_and_predict_deep(model_factory, train_idx, test_idx):
    # Normalizacja jest wyznaczana wyłącznie z osób treningowych.
    mean = X_sequence[train_idx].mean(axis=(0, 1), keepdims=True)
    std = X_sequence[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6
    x_train = (X_sequence[train_idx] - mean) / std
    x_test = (X_sequence[test_idx] - mean) / std
    x_train = torch.tensor(x_train.transpose(0, 2, 1), dtype=torch.float32)
    x_test = torch.tensor(x_test.transpose(0, 2, 1), dtype=torch.float32)
    y_train = torch.tensor(y_sequence[train_idx], dtype=torch.long)

    counts = np.bincount(y_sequence[train_idx], minlength=len(deep_classes))
    weights = len(y_train) / (len(deep_classes) * np.maximum(counts, 1))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    model = model_factory(len(RAW_CHANNELS), len(deep_classes)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

    model.train()
    for _ in range(DEEP_EPOCHS):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x.to(DEVICE)), batch_y.to(DEVICE))
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        predicted = model(x_test.to(DEVICE)).argmax(dim=1).cpu().numpy()
    return predicted, model, mean.squeeze(), std.squeeze()

deep_factories = {'1D CNN': CNN1D, 'TCN': TCN}
deep_predictions, deep_rows = {}, []
deep_splitter = LeaveOneGroupOut()
for model_name, factory in deep_factories.items():
    print(f'\n=== {model_name} — walidacja po osobach ===')
    start_time = perf_counter()
    predicted_indices = np.empty(len(y_sequence), dtype=np.int64)
    for fold, (train_idx, test_idx) in enumerate(deep_splitter.split(X_sequence, y_sequence, deep_groups), start=1):
        print(f'Runda {fold}/{len(np.unique(deep_groups))}: test dla {deep_groups[test_idx][0]}')
        predicted_indices[test_idx], _, _, _ = train_and_predict_deep(factory, train_idx, test_idx)
    predicted_labels = np.array([deep_classes[index] for index in predicted_indices])
    deep_predictions[model_name] = predicted_labels
    deep_report = pd.DataFrame(classification_report(sequence_labels, predicted_labels, output_dict=True, zero_division=0)).transpose()
    deep_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(sequence_labels, predicted_labels),
        'macro_f1': deep_report.loc['macro avg', 'f1-score'],
        'weighted_f1': deep_report.loc['weighted avg', 'f1-score'],
        'czas_treningu_s': perf_counter() - start_time,
    })

deep_comparison = pd.DataFrame(deep_rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
REPORTS_DIR.mkdir(exist_ok=True)
deep_comparison.to_csv(REPORTS_DIR / 'deep_model_comparison.csv', index=False)
all_model_comparison = pd.concat([pd.read_csv(REPORTS_DIR / 'model_comparison.csv'), deep_comparison], ignore_index=True)
all_model_comparison = all_model_comparison.sort_values('macro_f1', ascending=False).reset_index(drop=True)
all_model_comparison.to_csv(REPORTS_DIR / 'all_model_comparison.csv', index=False)
display(all_model_comparison.style.format({'accuracy': '{:.3f}', 'macro_f1': '{:.3f}', 'weighted_f1': '{:.3f}', 'czas_treningu_s': '{:.1f}'}))